# S03 · 03 — HPO con runs anidados, Model Registry y model card

El notebook 02 dejó corridas comparables. Ahora faltan tres cosas para poder
decir que un modelo "está en producción":

1. Buscar hiperparámetros de forma **sistemática** y que el resultado siga siendo
   navegable (no 200 runs sueltos).
2. Un **nombre estable** al que apunte el resto del sistema, que no sea un
   `run_id`.
3. **Documentación generada**, no escrita a mano.

## Antes de empezar

- El servidor de MLflow en `:5001` (`make mlflow`).
- Las particiones materializadas (`make data`).

> **Sobre el tiempo:** la búsqueda de hiperparámetros es la celda más lenta de la
> sesión. Lánzala al empezar el bloque y sigue leyendo mientras corre.
> **Cronométrala en tu máquina** en lugar de creerle a un número escrito aquí: el
> costo depende de tus núcleos y de la cardinalidad de tus features.

In [ ]:
import subprocess
import sys

import mlflow

from taxi import config
from taxi.models import evaluate, registry, train

mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
cliente = registry.cliente()
print("tracking URI:", mlflow.get_tracking_uri())
print("modelo registrado del curso:", config.MODELO_REGRESION)
print("URI canonica de produccion:", config.uri_modelo())

## 1. HPO: por qué 20 intentos informados le ganan a 200 al azar

Optuna no prueba combinaciones a ciegas: modela la relación entre
hiperparámetros y métrica (`TPESampler`) y concentra los intentos en la zona
prometedora. Además puede **abandonar** un intento que ya se ve peor que la
mediana (`MedianPruner`), lo que devuelve tiempo de cómputo al presupuesto.

Dos decisiones de diseño que hay que ver, porque son las que hacen la diferencia
entre una UI legible y un experimento inservible:

| Decisión | Por qué |
|---|---|
| **Un parent run para el study, un child run por trial** (`nested=True`) | 200 runs sueltos no se pueden agrupar ni comparar. En el parent quedan el espacio de búsqueda y los mejores params; en cada child, su combinación y su métrica |
| **Semilla explícita en el sampler** | sin ella, "mi mejor RMSE fue 4.31" no es una afirmación reproducible |
| **El objetivo se mide en `PARTICION_VALID`, nunca en el holdout** | la búsqueda vería el holdout cientos de veces y el gate de S06 dejaría de medir generalización |

Esto ya está implementado en `train.optimizar_hiperparametros`. Léelo: son 60
líneas y la mitad son comentarios que explican por qué.

In [ ]:
# El taller pide >=20 runs en el experimento de HPO. Aqui se usan pocos para que
# la clase avance; para los 20 del taller: `make hpo` (taxi train --hpo --trials 20).
TRIALS = 5

resultado = train.optimizar_hiperparametros(trials=TRIALS, registrar_mejor=True)

print("run del mejor modelo:", resultado.run_id)
print("valid_rmse:", round(resultado.rmse_valid, 4))
print("version registrada:", resultado.version_registrada)

### Qué mirar en la UI

1. El experimento `s03-hpo-xgboost` tiene **un** run de nivel superior por
   study; los trials están anidados debajo.
2. En el parent: `trials`, `espacio`, `trials_completados` y `trials_podados`. Si
   `trials_podados` es 0 con muchos trials, el pruner no está actuando.
3. En cada child: `mejor_iteracion`. Si sale siempre igual a `n_estimators - 1`,
   el early stopping tampoco está actuando.

## 2. Resolver un `run_id` con código, nunca a mano

La versión anterior de este notebook traía esto:

```python
run_id = "7dcc1f0c8901478892758fc00fcb18e1"  # reemplazar con el run_id del experimento
client.list_artifacts(run_id)
```

Ese `run_id` era el de la máquina de quien escribió el notebook. Para cualquier
otra persona, la celda falla con un `RestException: Run ... not found`, y el
comentario "reemplazar con el run_id" traslada al estudiante un trabajo que el
código puede hacer solo.

La forma correcta es preguntarle al tracking server.

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[config.EXPERIMENTOS["hpo"]],
    filter_string="tags.tipo = 'candidato'",
    order_by=["metrics.valid_rmse ASC"],
    max_results=5,
)

if runs.empty:
    print("No hay runs todavia. Corre la celda de HPO o `make hpo`.")
else:
    run_id = runs.loc[0, "run_id"]
    print("mejor run:", run_id, "valid_rmse:", round(runs.loc[0, "metrics.valid_rmse"], 4))
    print("\nartifacts del run:")
    for artefacto in cliente.list_artifacts(run_id):
        print("  ", artefacto.path)
    print("\ndentro de 'modelo':")
    for artefacto in cliente.list_artifacts(run_id, path="modelo"):
        print("  ", artefacto.path)

Los artifacts del run del candidato incluyen el modelo, las figuras y el JSON de
métricas por subgrupo. Fíjate en `modelo/MLmodel`: ahí están la `signature`, el
`input_example` y el flavor. Es el archivo que responde "¿esto se puede servir?".

## 3. Model Registry: registrar no es promover

Un run es una **ejecución**. Una versión del registry es un **artefacto con
nombre**, al que el resto del sistema puede apuntar sin saber de qué run salió.

```mermaid
flowchart LR
    T["Tracking<br/>s03-hpo-xgboost<br/>runs y artifacts"] -->|"register_model"| REG[("Registry<br/>nyc-taxi-duration<br/>v1, v2, v3 (inmutables)")]
    REG -->|"alias @candidate<br/>validation_status=pending"| GATE{"Gate en CI (S06)<br/>holdout fijo"}
    GATE -->|"mejora"| CH["alias @champion<br/>validation_status=passed"]
    GATE -->|"no mejora"| NO["no promueve"]
    CH --> API["API y batch<br/>models:/nyc-taxi-duration@champion"]
```

**Aliases para enrutar, tags para documentar.** Es la decisión de
[`../../../docs/adr/002-aliases-en-vez-de-stages.md`](../../../docs/adr/002-aliases-en-vez-de-stages.md),
y el resumen vive en el código para que el notebook y la model card impriman la
**misma** explicación.

In [ ]:
print(registry.explicar_por_que_no_stages())

In [ ]:
candidato = registry.version_por_alias(config.MODELO_REGRESION, config.ALIAS_CANDIDATO)
champion = registry.version_por_alias(config.MODELO_REGRESION, config.ALIAS_PRODUCCION)

print("candidate:", candidato.version if candidato else None)
print("champion :", champion.version if champion else None)
print("tags del candidato:", candidato.tags if candidato else {})
print("\ntodos los aliases:", cliente.get_registered_model(config.MODELO_REGRESION).aliases)

> `champion` en `None` es un estado **normal**: es el estado del primer día. Por
> eso `registry.version_por_alias` devuelve `None` en lugar de propagar la
> excepción — el gate tiene que poder distinguir "todavía no hay champion" de
> "no pude hablar con MLflow".
>
> Y fíjate en cómo se resolvió "la última versión": con `search_model_versions`,
> no con `get_latest_versions`. El segundo está deprecado porque su semántica era
> "la última de cada *stage*", y los stages ya no existen.

## 4. Promover: el tag primero, el alias después

El orden no es un detalle de estilo. Si el proceso muere entre las dos
operaciones:

- tag → alias: queda "validada pero no promovida". Seguro.
- alias → tag: queda un modelo **sirviendo tráfico sin registro de haber sido
  validado**. Es exactamente el incidente que el gate existe para evitar.

En esta sesión promovemos a mano para ver el mecanismo. En S06 lo hace el gate de
CI (`scripts/promote.py`), que además compara contra el holdout fijo y por
subgrupos: aquí no hay ningún criterio, solo la mecánica.

In [ ]:
version = candidato.version if candidato else resultado.version_registrada

registry.marcar_validacion(config.MODELO_REGRESION, version, "passed")
registry.asignar_alias(config.MODELO_REGRESION, config.ALIAS_PRODUCCION, version)

print("aliases ahora:", cliente.get_registered_model(config.MODELO_REGRESION).aliases)
print("tags de la version:", cliente.get_model_version(config.MODELO_REGRESION, version).tags)

## 5. Cargar por alias y reproducir la métrica

Las tres formas de referirse a un modelo del registry:

| Forma | URI | Cuándo |
|---|---|---|
| Por versión | `models:/nyc-taxi-duration/7` | debugging, auditoría, reproducir un incidente |
| **Por alias** | `models:/nyc-taxi-duration@champion` | **producción**: el código no cambia cuando cambia la versión |
| Por stage | `models:/nyc-taxi-duration/Production` | **no usar**: los stages están deprecados desde MLflow 2.9.0 |

Y se carga como **`pyfunc`**, no con el flavor nativo: quien consume el modelo
—la API, el batch, el gate— no debería tener que saber con qué librería se
entrenó. Si mañana el champion pasa de sklearn a XGBoost, ese código no cambia.

La celda siguiente es el **criterio de aceptación 1 del taller**: cargar desde
`@champion` y reproducir la métrica que reportó el run.

In [ ]:
modelo = registry.cargar_por_alias()  # models:/nyc-taxi-duration@champion

df_valid = train.cargar_valid()
metricas, subgrupos = evaluate.evaluar_modelo(modelo, df_valid, prefijo="valid_")

registradas = registry.metricas_de_version(config.MODELO_REGRESION, version)
rmse_run = registradas.get("valid_rmse", float("nan"))
rmse_reproducido = metricas["valid_rmse"]

print(f"valid_rmse en el run     : {rmse_run:.6f}")
print(f"valid_rmse reproducido   : {rmse_reproducido:.6f}")
print(f"diferencia absoluta      : {abs(rmse_run - rmse_reproducido):.2e}")
print("\nRMSE por subgrupo (los que el promedio esconde):")
for nombre in sorted(k for k in subgrupos if k.startswith("rmse_")):
    print(f"  {nombre:22s} {subgrupos[nombre]:.4f}")

Si los dos números coinciden hasta el último decimal, tienes reproducibilidad de
la métrica: mismo modelo, mismos datos, mismo código de evaluación.

Cuándo **no** coinciden, y qué significa cada caso:

| Diferencia | Causa probable |
|---|---|
| exacta (0.0) | todo bien |
| ~1e-6 | orden de operaciones en coma flotante o distinto número de hilos |
| decimales visibles | evaluaste otra partición, otro subconjunto, u otra definición de la métrica |
| enorme | estás cargando otro modelo (revisa a qué versión apunta el alias) |

Por eso el taller pide **declarar la tolerancia**: un criterio de aceptación sin
tolerancia declarada no se puede verificar.

### Rollback

Volver atrás es mover el alias a la versión anterior:

```python
registry.asignar_alias("nyc-taxi-duration", "champion", "6")
```

Una escritura de metadatos: sub-segundo, sin reentrenar, sin reconstruir imagen,
sin redeploy. Funciona porque **las versiones del registry son inmutables**: la
versión anterior sigue ahí, es la evidencia de lo que estuvo en producción.

## 6. Contraejemplo: la API de stages (no usar)

Esto **no es** lo que hay que hacer. Está aquí porque vas a encontrarlo en la
mayoría de los tutoriales de la web y en el código que heredes, y hay que saber
reconocerlo y traducirlo.

```
None  →  Staging  →  Production  →  Archived
```

Los stages están **deprecados desde MLflow 2.9.0**. El método del cliente todavía
existe (verificado en 3.15.1) y la documentación oficial anuncia su eliminación
en una versión mayor. Los cuatro problemas de fondo:

1. Vocabulario **cerrado** de cuatro palabras. Un equipo real necesita más:
   `champion`, `challenger`, `shadow`, `canary`, `champion-eu`.
2. Mezclaba dos cosas distintas: qué versión sirve (*routing*) y en qué estado de
   validación está (*metadato*). Aliases y tags las separan.
3. **Dos versiones podían quedar en `Production` a la vez** y nadie sabía cuál
   respondía. Un alias apunta a una sola versión, siempre.
4. `archive_existing_versions=True` archivaba la versión anterior de forma
   automática: el rollback dejaba de ser trivial.

La traducción, término a término:

| API de stages (no usar) | Equivalente vigente |
|---|---|
| `client.transition_model_version_stage(n, v, stage="Production")` | `client.set_registered_model_alias(n, "champion", v)` |
| `client.get_latest_versions(n, stages=["Production"])` | `client.get_model_version_by_alias(n, "champion")` |
| `models:/<n>/Production` | `models:/<n>@champion` |
| `stage="Staging"` como "en validación" | tag `validation_status=pending` |

La celda siguiente está desactivada a propósito. Actívala solo si quieres ver el
`DeprecationWarning` con tus propios ojos; en el material del curso, el
`pre-commit` bloquea estas llamadas fuera de `notebooks/` y `scenarios/`.

In [ ]:
EJECUTAR_CONTRAEJEMPLO = False  # ponlo en True solo para ver el warning

if EJECUTAR_CONTRAEJEMPLO:
    # CONTRAEJEMPLO. No copiar a codigo real: API deprecada desde MLflow 2.9.0.
    cliente.transition_model_version_stage(
        name=config.MODELO_REGRESION,
        version=version,
        stage="Staging",
        archive_existing_versions=False,
    )
    print("stage:", cliente.get_model_version(config.MODELO_REGRESION, version).current_stage)
else:
    print("Contraejemplo desactivado. La via vigente es:")
    print(f"  registry.asignar_alias('{config.MODELO_REGRESION}', 'champion', '{version}')")

## 7. Model card: documentación generada, no escrita

Una model card escrita a mano queda desactualizada en el primer reentrenamiento.
`scripts/model_card.py` la genera **desde los metadatos del registry**: versión,
run de origen, particiones con su SHA-256, métricas globales y por subgrupo,
limitaciones y uso no previsto.

Eso la convierte en algo que se puede **verificar**: si el modelo cambió y la card
no, es porque nadie corrió el generador, y eso se ve en el diff.

Es también el vehículo práctico de la documentación técnica que exige el AI Act
(S07 lo retoma con la gobernanza completa).

In [ ]:
proceso = subprocess.run(
    [sys.executable, "scripts/model_card.py"],
    cwd=config.PROJECT_ROOT,
    capture_output=True,
    text=True,
)
print(proceso.stdout or proceso.stderr)

card = config.PROJECT_ROOT / "docs" / "model-card.md"
print("\n".join((card.read_text(encoding="utf-8")).splitlines()[:35]))

> El mismo comando, desde la terminal: `make model-card`. Es el que pide el
> criterio de aceptación 3 del taller.

## 8. Lo que esta sesión deliberadamente NO hace

- **No decide** si el candidato merece ser champion. Aquí se promovió a mano para
  ver el mecanismo; el criterio (holdout fijo, mejora mínima, chequeo por
  subgrupos) es el gate de S06 en `scripts/promote.py`. Pruébalo con
  `uv run taxi promote --dry-run`: no escribe nada y muestra la tabla de
  decisión.
- **No sirve** el modelo por HTTP: eso es S05.
- **No mira el holdout.** `PARTICION_TEST` sigue sin usarse, y así debe seguir
  hasta el gate.

## Siguiente

[`../taller.md`](../taller.md) — el mismo flujo, sobre tu propio proyecto, con
criterios de aceptación medibles.